In [26]:
import dotenv
from google.adk.runners import InMemoryRunner
from google.genai.types import Part
from google.genai.types import UserContent
from llm_auditor.agent import root_agent, critic_agent, llm_auditor
import textwrap


In [2]:
dotenv.load_dotenv()

True

In [3]:
from llm_auditor.sub_agents.critic.agent import prompt

In [ ]:
runner = InMemoryRunner(agent=critic_agent)

def create_verification_prompt(claim: str) -> str:
    return f"Verify this claim: {claim}"

async def evaluate(claim: str) -> prompt.CriticOutput:
    """
    Evaluates a given claim using a runner session.

    Args:
        claim: The claim string to be evaluated.

    Returns:
        The structured CriticOutput response from the evaluation.
    """
    session = runner.session_service.create_session(
        app_name=runner.app_name, user_id="test_user"
    )
    content = UserContent(parts=[Part(text=create_verification_prompt(claim))])
    events = []
    async for event in runner.run_async(
        user_id=session.user_id, session_id=session.id, new_message=content
    ):
        events.append(event)

    raw_text = events[-1].content.parts[0].text

    return prompt.CriticOutput.model_validate_json(raw_text)

In [ ]:
# """Runs the agent on a simple input and expects a normal response."""
# claim = textwrap.dedent("""
#     Verify this claim:
#     Blue Ridge Outfitters is headquartered in a small village far from any city.
# """).strip()

# response = await evaluate(claim=claim)
# response


In [40]:
auditor_runner = InMemoryRunner(agent=llm_auditor)

# def create_verification_prompt(claim: str) -> str:
#     return f"Verify this claim: {claim}"

async def revise_claim(claim: str) -> str:
    """
    Revises a claim using a runner session.

    Args:
        claim: The claim string to be evaluated.

    Returns:
        The rewritten claim.
    """
    session = auditor_runner.session_service.create_session(
        app_name=runner.app_name, user_id="test_user"
    )
    content = UserContent(parts=[Part(text=create_verification_prompt(claim))])
    events = []
    async for event in auditor_runner.run_async(
        user_id=session.user_id, session_id=session.id, new_message=content
    ):
        events.append(event)

    raw_text = events[-1].content.parts[0].text

    return raw_text

    # return prompt.CriticOutput.model_validate_json(raw_text)

In [41]:
response = await revise_claim(claim=claim)
response

'Blue Ridge Outfitters is headquartered in Asheville, North Carolina. This location serves as its corporate home and a critical logistics base, supporting its retail stores, online presence, and guided adventures. The headquarters is located at 1 Appalachian Way, Asheville, NC. The main headquarters building encompasses 50,000 square feet and houses essential corporate functions.\n'

In [7]:
import pathlib
from llm_auditor.claims import Claim
from typing import List
import yaml
import glob

def load_claims_from_cache(cache_dir: pathlib.Path) -> List[Claim]:
    """
    Recursively loads Claim objects from all YAML files in the given cache directory.

    Args:
        cache_dir: The path to the cache directory.

    Returns:
        A list of Claim objects loaded from the cache.

    Raises:
        FileNotFoundError: If the cache directory does not exist.
        yaml.YAMLError: If there's an error decoding a YAML file.
        OSError: If there's a problem reading a file.
    """
    if not cache_dir.exists() or not cache_dir.is_dir():
        raise FileNotFoundError(f"Cache directory not found: {cache_dir}")

    all_claims: List[Claim] = []
    yaml_files = glob.glob(str(cache_dir / "**" / "*.yaml"), recursive=True)
    for file_path_str in yaml_files:
        file_path = pathlib.Path(file_path_str)
        try:
            text = file_path.read_text()
            if not text:
                continue
            data = yaml.load(text, Loader=yaml.SafeLoader)
            if data is not None:
                for item in data:
                    all_claims.append(Claim(**item))
        except yaml.YAMLError as e:
            print(f"Error decoding YAML file: {file_path}. Error: {e}")
            raise
        except OSError as e:
            print(f"Error reading file: {file_path}. Error: {e}")
            raise

    return all_claims


In [8]:
from pathlib import Path
import yaml

claims_dir_path = Path("claims/claims.yaml")
claims = load_claims_from_cache(cache_dir=pathlib.Path("/Users/ivanmkc/code/adk-samples/output/claim_cache/with_contexts"))

print(len(claims))

1206


In [9]:
claims = [claim for claim in claims if claim.is_supported == claim.is_supported_after_rewriting and claim.source_contains_context]

print(len(claims))

570


In [10]:
import asyncio
from tqdm.asyncio import tqdm

async def _evaluate_single_claim(claim: Claim, semaphore: asyncio.Semaphore) -> prompt.CriticOutput | None:
    """
    Evaluates a single claim using the 'evaluate' async function.
    Includes concurrency control and basic error handling.

    Args:
        claim: The Claim object to evaluate.
        semaphore: An asyncio.Semaphore to limit concurrent access.

    Returns:
        The evaluated Claim object, or None if an error occurs.
    """
    async with semaphore:
        try:
            # Call the actual evaluation function
            result = await evaluate(claim=claim.claim)
            return result
        except Exception as e:
            print(f"An error occurred during evaluation of '{claim.claim}': {e}")
            return None

async def evaluate_claims_async(claims: list[Claim], max_concurrency: int = 5) -> list[Claim | None]:
    """
    Asynchronously evaluates a list of Claim objects with concurrency control
    and a progress bar.

    Args:
        claims: A list of Claim dataclass instances to evaluate.
        max_concurrency: The maximum number of concurrent evaluation tasks.

    Returns:
        A list of evaluated Claim objects, or None if the evaluation failed for that claim.
    """
    semaphore = asyncio.Semaphore(max_concurrency)
    tasks = [_evaluate_single_claim(claim, semaphore) for claim in claims]

    print(f"\nStarting asynchronous evaluation of {len(claims)} claims (max concurrency: {max_concurrency})...")
    # Use tqdm.asyncio.tqdm.gather for concurrent execution with a progress bar
    results = await tqdm.gather(*tasks, desc="Evaluating Claims")
    print("Asynchronous evaluation complete.")
    return results

In [43]:
async def _revise_single_claim(claim: Claim, semaphore: asyncio.Semaphore) -> str | None:
    """
    Revises a single claim using the 'revise_claim' async function.
    Includes concurrency control and basic error handling.

    Args:
        claim: The Claim object to evaluate.
        semaphore: An asyncio.Semaphore to limit concurrent access.

    Returns:
        The revised claim str, or None if an error occurs.
    """
    async with semaphore:
        try:
            # Call the actual evaluation function
            result = await revise_claim(claim=claim.claim)
            return result
        except Exception as e:
            print(f"An error occurred during evaluation of '{claim.claim}': {e}")
            return None

async def revise_claims_async(claims: list[Claim], max_concurrency: int = 5) -> list[str | None]:
    """
    Asynchronously evaluates a list of Claim objects with concurrency control
    and a progress bar.

    Args:
        claims: A list of Claim dataclass instances to evaluate.
        max_concurrency: The maximum number of concurrent evaluation tasks.

    Returns:
        A list of revised claim str's, or None if the revision failed for that claim.
    """
    semaphore = asyncio.Semaphore(max_concurrency)
    tasks = [_revise_single_claim(claim, semaphore) for claim in claims]

    print(f"\nStarting asynchronous evaluation of {len(claims)} claims (max concurrency: {max_concurrency})...")
    # Use tqdm.asyncio.tqdm.gather for concurrent execution with a progress bar
    results = await tqdm.gather(*tasks, desc="Revising Claims")
    print("Asynchronous revision complete.")
    return results

In [11]:
results = await evaluate_claims_async(claims=claims)

assert len(results) == len(claims)


Starting asynchronous evaluation of 570 claims (max concurrency: 5)...


Evaluating Claims:  55%|█████▍    | 311/570 [05:52<15:55,  3.69s/it]

An error occurred during evaluation of 'Kayaks are stocked year-round at the Asheville and Bend locations.': 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}


Evaluating Claims:  71%|███████   | 402/570 [07:41<02:08,  1.31it/s]

An error occurred during evaluation of 'An assessment of adherence to Leave No Trace principles is not performed.': 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Request contains an invalid argument.', 'status': 'INVALID_ARGUMENT'}}


Evaluating Claims: 100%|██████████| 570/570 [10:42<00:00,  1.13s/it]

Asynchronous evaluation complete.


In [45]:
revised_claims = await revise_claims_async(claims=claims)

assert len(revised_claims) == len(claims)


Starting asynchronous evaluation of 570 claims (max concurrency: 5)...


Revising Claims:  22%|██▏       | 123/570 [03:31<47:50,  6.42s/it]

An error occurred during evaluation of 'The first Blue Ridge Outfitters store was larger than 800 square feet.': 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}


Revising Claims:  57%|█████▋    | 323/570 [08:00<09:46,  2.37s/it]

An error occurred during evaluation of 'The sessions held at Flagship Blue Ridge Outfitters are exclusively for paying customers and are not open to the general local community.': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}


Revising Claims:  57%|█████▋    | 325/570 [08:06<10:01,  2.45s/it]

An error occurred during evaluation of 'The design team at Pathfinder Gear is led by a person named Sarah Jenkins.': 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}


Revising Claims: 100%|██████████| 570/570 [14:03<00:00,  1.48s/it]

Asynchronous revision complete.


In [54]:
import pandas as pd

LOAD_RESULTS_FROM_CACHE = False

def extract_verdict(output: prompt.CriticOutput) -> str:
    return output.overall_assessment.overall_verdict

if LOAD_RESULTS_FROM_CACHE:
    # Load from cache
    df = pd.read_csv("critic_output.csv")
else:
    # results = await evaluate_claims_async(claims=claims)
    verdicts = [extract_verdict(result) if result else None for result in results]
    
    df = pd.DataFrame([dict(
        claim=claim.claim, 
        revised_claim=revised_claim, 
        context=claim.context, 
        is_supported=claim.is_supported, 
        verdict=verdict)
        for claim, verdict, revised_claim in zip(claims, verdicts, revised_claims)
        if claim.is_supported == claim.is_supported_after_rewriting and claim.source_contains_context
        ])
    df = df.dropna()

    # Save to cache
    df.to_csv("critic_output.csv", index=False)

    # assert pd.read_csv("critic_output.csv") == df

assert len(results) == len(claims)

In [50]:
# Convert overall verdict to a boolean prediction to compare with
df['is_supported_predicted'] = df['verdict'].apply(lambda x: x.lower().strip() == "accurate")

df.head()

,claim,revised_claim,is_supported,verdict,is_supported_predicted
0,Blue Ridge Outfitters' flagship stores are loc...,Blue Ridge Outfitters' flagship stores are loc...,False,Accurate,True
1,The flagship stores for Blue Ridge Outfitters ...,The flagship stores for Blue Ridge Outfitters ...,False,Accurate,True
2,"The strategy behind the location, design, and ...","The strategy behind the location, design, and ...",False,Inaccurate,False
3,The placement of each flagship store was a ran...,The placement of each flagship store was a del...,False,Inaccurate,False
4,The selection of flagship store locations was ...,This claim is inaccurate. The selection of fla...,False,Inaccurate,False


In [51]:
df.verdict.unique()

array(['Accurate', 'Inaccurate', 'Unsupported', 'Partially Accurate'],
      dtype=object)

In [52]:
df[df['is_supported'] != df['is_supported_predicted']].head()

,claim,revised_claim,is_supported,verdict,is_supported_predicted
0,Blue Ridge Outfitters' flagship stores are loc...,Blue Ridge Outfitters' flagship stores are loc...,False,Accurate,True
1,The flagship stores for Blue Ridge Outfitters ...,The flagship stores for Blue Ridge Outfitters ...,False,Accurate,True
5,Flagship stores were intentionally situated in...,Flagship stores were intentionally situated in...,False,Accurate,True
6,The Bend store was placed in a remote industri...,The Bend store was strategically placed within...,False,Accurate,True
7,The Jackson store's location was chosen to exc...,The Jackson store's location was not chosen to...,False,Accurate,True


In [23]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Extract the actual and predicted values
y_actual = df['is_supported']
y_predicted = df['is_supported_predicted']

print(f"Support: {len(y_actual)}")

# 1. Accuracy Score
# The proportion of correctly classified instances.
accuracy = accuracy_score(y_actual, y_predicted)
print(f"Accuracy: {accuracy:.4f}")

# 2. Precision Score
# Precision (of the false class): TP / (TP + FP)
# How many of the predicted positives were actually positive?
# 'pos_label=True' explicitly sets True as the positive class.
precision = precision_score(y_actual, y_predicted, pos_label=False)
print(f"Precision (for False): {precision:.4f}")

# 3. Recall Score (Sensitivity or True Positive Rate)
# Recall (of the positive class): TP / (TP + FN)
# How many of the actual positives were correctly predicted?
recall = recall_score(y_actual, y_predicted, pos_label=False)
print(f"Recall (for False): {recall:.4f}")

# 4. F1-Score
# The harmonic mean of Precision and Recall.
# A good balance between precision and recall.
f1 = f1_score(y_actual, y_predicted, pos_label=False)
print(f"F1-Score (for False): {f1:.4f}")

# 5. Confusion Matrix
# A table showing True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN).
# The order is usually: TN | FP
#                       FN | TP
# By default, sklearn's confusion_matrix assumes 0 is negative and 1 is positive.
# If your booleans are treated as 0s and 1s (False=0, True=1), then:
# cm[0,0] = TN (actual False, predicted False)
# cm[0,1] = FP (actual False, predicted True)
# cm[1,0] = FN (actual True, predicted False)
# cm[1,1] = TP (actual True, predicted True)
conf_matrix = confusion_matrix(y_actual, y_predicted)
print("\nConfusion Matrix:")
print(conf_matrix)


Support: 568
Accuracy: 0.8063
Precision (for False): 0.9892
Recall (for False): 0.8128
F1-Score (for False): 0.8924

Confusion Matrix:
[[456 105]
 [  5   2]]


In [24]:
df.columns

Index(['claim', 'is_supported', 'verdict', 'is_supported_predicted'], dtype='object')

In [18]:
#  TODO Plot histogram of results to see if it's too heavy on the ends

# df.value_counts(['is_supported_predicted'])
sum(result is None for result in results)

2